# Synthetic Q&A Generation — UR5e Cobot + Bridgeport Mill

Generates 100 new Q&A pairs (50 per machine) by:
- Randomly sampling example Q&A pairs from the golden sets as few-shot examples
- Randomly sampling text chunks from the cropped PDFs as grounding context
- Using the LLM to generate new, diverse Q&A pairs grounded in the manuals

**Output:** `data/QA/synthetic_qa_cobot_mill.csv` — columns: `machine, question, gold_answer`

In [1]:
import sys
import random
import json
import re
from pathlib import Path

import pandas as pd
from openai import OpenAI
from langchain_community.document_loaders import PyMuPDFLoader

sys.path.insert(0, str(Path("..").resolve()))

from system.rag.rag_utils import _format_sources_xml
from system.utils import EnvironmentConfig
from system.preprocess import PDFPreprocessor

/Users/ryan/Desktop/Work/SIGHT/safety_rag_eval_ryan/venv/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Load Golden Sets

In [2]:
COBOT_QA_PATH = Path("../data/QA/COBOT/Final_COBOT_QA.csv")
MILL_QA_PATH  = Path("../data/QA/MILL/Mill Feedback Accepted.csv")

cobot_golden = (
    pd.read_csv(COBOT_QA_PATH)
    .dropna(subset=["question", "gold_answer"])
    .reset_index(drop=True)
)
mill_golden = (
    pd.read_csv(MILL_QA_PATH)
    .dropna(subset=["question", "gold_answer"])
    .reset_index(drop=True)
)

print(f"COBOT golden set : {len(cobot_golden)} rows")
print(f"Mill  golden set : {len(mill_golden)} rows")
display(cobot_golden.head(3))
display(mill_golden.head(3))

COBOT golden set : 32 rows
Mill  golden set : 51 rows


,question,gold_answer
0,What safety modes does the robot arm have?,The robot arm has three safety modes. Normal m...
1,What happens if my robot stopped in 1000ms whe...,The robot arm will enter recovery mode because...
2,What mode do I need to be in to load a program?,You need to be in Automatic Mode to load and e...


,question,gold_answer
0,What are the 5 main categories of safety label...,DANGER\nDANGER indicates a hazardous situation...
1,What are the safety labels used in the birdgep...,DANGER(Red with white text)\nDANGER indicates ...
2,I am trying to mount the Spindle Guard onto th...,There are two tapped holes in the nose cap of ...


## Crop PDFs + Extract Text

Reused from `system/main_long_context.ipynb` — same crop parameters.

In [3]:
PDF_DIR = Path("../data/input/input_pdfs")

TARGETS = {
    "Mill (Bridgeport)": {
        "path": PDF_DIR / "Bridgeport Series 1 Milling manual with schematics.pdf",
        "crop_top": 0.04, "crop_bottom": 0.075, "crop_left": 0.0, "crop_right": 0.0,
    },
    "UR5e Cobot": {
        "path": PDF_DIR / "UR5e_Universal_Robots User Manual.pdf",
        "crop_percent": 0.075,
    },
}

def load_pdf_as_text(path) -> str:
    pages = PyMuPDFLoader(str(path)).load()
    return "\n\n".join(p.page_content for p in pages).strip()

OUTPUT_DIR = Path("../data/input/cropped_pdfs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CROPPED = {}
for label, cfg in TARGETS.items():
    out = OUTPUT_DIR / f"cropped_{cfg['path'].name}"
    PDFPreprocessor.crop_pdf(
        cfg["path"], out,
        crop_percent=cfg.get("crop_percent", 0.075),
        crop_top=cfg.get("crop_top"),
        crop_bottom=cfg.get("crop_bottom"),
        crop_left=cfg.get("crop_left"),
        crop_right=cfg.get("crop_right"),
    )
    CROPPED[label] = out
    print(f"Cropped {label} -> {out.name}")

mill_cropped_text = load_pdf_as_text(CROPPED["Mill (Bridgeport)"])
ur5e_cropped_text = load_pdf_as_text(CROPPED["UR5e Cobot"])

print(f"\nMill  (cropped): {len(mill_cropped_text):,} chars")
print(f"UR5e  (cropped): {len(ur5e_cropped_text):,} chars")

Cropped Mill (Bridgeport) -> cropped_Bridgeport Series 1 Milling manual with schematics.pdf
Cropped UR5e Cobot -> cropped_UR5e_Universal_Robots User Manual.pdf

Mill  (cropped): 121,502 chars
UR5e  (cropped): 253,244 chars


## Chunk Document Text

Split each cropped PDF into overlapping ~2000-char chunks. Each chunk acts as a retrieved page that the LLM uses as grounding context.

In [4]:
def chunk_text(text: str, chunk_size: int = 2000, overlap: int = 200) -> list:
    """Split text into overlapping fixed-size chunks."""
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start : start + chunk_size])
        start += chunk_size - overlap
    return chunks

mill_chunks  = chunk_text(mill_cropped_text)
ur5e_chunks  = chunk_text(ur5e_cropped_text)

print(f"Mill  chunks : {len(mill_chunks)}")
print(f"UR5e  chunks : {len(ur5e_chunks)}")

Mill  chunks : 68
UR5e  chunks : 141


## Synthetic QA Generation

Each call to `generate_synthetic_qa()` independently randomizes:
- **`n_examples`** example Q&A pairs drawn from the golden set (few-shot style)
- **`n_chunks`** text chunks drawn from the PDF (grounding context)

Chunks are formatted via `_format_sources_xml()` (reused from `system/rag/rag_utils.py`) so the LLM sees the same structured XML format used during evaluation.

In [8]:
SYNTH_SYSTEM_PROMPT = """\
You are an expert at creating safety and operational Q&A pairs for industrial equipment manuals.
You will be given:
  1. Example Q&A pairs that show the expected style and scope.
  2. Source passages from the machine manual.

Your task: generate {n} NEW question-answer pairs.

Rules:
- Every question must be directly answerable from the provided source passages.
- Answers must be concise (1–5 sentences or short bullet points), mirroring manual terminology.
- Do NOT duplicate or closely paraphrase the example questions.
- Question/answer pairs should range from easy to hard. 
- Cover diverse topics: safety procedures, operating steps, warnings, maintenance, specifications.
- Output ONLY a valid JSON array with no extra text:
  [{{"question": "...", "gold_answer": "..."}}, ...]
"""


def generate_synthetic_qa(
    client: OpenAI,
    document_chunks: list,
    golden_df: pd.DataFrame,
    machine_name: str,
    n_examples: int = 6,
    n_chunks: int = 5,
    n_generate: int = 5,
    model: str = "gpt-5.4-2026-03-05",
) -> list:
    """
    Generate `n_generate` synthetic Q&A pairs for `machine_name`.

    Randomizes both the few-shot examples drawn from the golden set
    and the document chunks used as grounding context on every call.

    Reuses `_format_sources_xml()` from system/rag/rag_utils.py to
    format the document chunks in the same XML structure used during
    RAG evaluation.
    """
    # 1. Randomly sample few-shot examples from the golden set
    examples = golden_df.sample(n=min(n_examples, len(golden_df)))
    examples_text = "\n\n".join(
        f"Q: {row.question}\nA: {row.gold_answer}"
        for _, row in examples.iterrows()
    )

    # 2. Randomly sample document chunks and build hits in the expected schema
    selected = random.sample(document_chunks, min(n_chunks, len(document_chunks)))
    hits = [
        {"filename": machine_name, "file_id": "", "score": None, "text": chunk}
        for chunk in selected
    ]

    # 3. Format chunks as XML using the existing rag_utils helper
    sources_xml = _format_sources_xml(hits, max_chars_per_content=2000)

    # 4. Build messages and call LLM (same pattern as _ask_with_sources in rag_utils.py)
    system_msg = SYNTH_SYSTEM_PROMPT.format(n=n_generate)
    user_msg = (
        f"Machine: {machine_name}\n\n"
        f"Example Q&A pairs (do NOT duplicate):\n{examples_text}\n\n"
        f"Generate {n_generate} new Q&A pairs based on the source passages below."
    )

    resp = client.responses.create(
        model=model,
        input=[
            {"role": "system",  "content": system_msg},
            {"role": "user",    "content": user_msg},
            {"role": "user",    "content": f"Source passages:\n{sources_xml}"},
        ],
        reasoning={"effort": "low"},
        max_output_tokens=3000,
    )

    # 5. Parse JSON array from response
    raw = getattr(resp, "output_text", "") or ""
    match = re.search(r"\[.*\]", raw, re.DOTALL)
    if not match:
        print(f"  [WARN] Could not parse JSON from response for {machine_name}")
        return []
    try:
        pairs = json.loads(match.group())
    except json.JSONDecodeError as e:
        print(f"  [WARN] JSON decode error: {e}")
        return []

    return [
        {
            "machine": machine_name,
            "question": p["question"],
            "gold_answer": p["gold_answer"],
        }
        for p in pairs
        if isinstance(p, dict) and "question" in p and "gold_answer" in p
    ]


print("generate_synthetic_qa() defined.")

generate_synthetic_qa() defined.


## Generate — UR5e Cobot (target: 50 pairs)

10 batches × 5 pairs per batch = 50 target.

In [11]:
client = OpenAI()

ur5e_results = []
for i in range(10):
    batch = generate_synthetic_qa(
        client,
        document_chunks=ur5e_chunks,
        golden_df=cobot_golden,
        machine_name="UR5e Cobot",
    )
    ur5e_results.extend(batch)
    print(f"  Batch {i+1:02d}: +{len(batch)} pairs  (running total: {len(ur5e_results)})")

print(f"\nUR5e Cobot total: {len(ur5e_results)} Q&A pairs generated")

  Batch 01: +5 pairs  (running total: 5)
  Batch 02: +5 pairs  (running total: 10)
  Batch 03: +5 pairs  (running total: 15)
  Batch 04: +5 pairs  (running total: 20)
  Batch 05: +5 pairs  (running total: 25)
  Batch 06: +5 pairs  (running total: 30)
  Batch 07: +5 pairs  (running total: 35)
  Batch 08: +5 pairs  (running total: 40)
  Batch 09: +5 pairs  (running total: 45)
  Batch 10: +5 pairs  (running total: 50)

UR5e Cobot total: 50 Q&A pairs generated


In [12]:
ur5e_results

[{'machine': 'UR5e Cobot',
  'question': 'What must be provided as part of the electrical installation for the UR5e Control Box mains connection?',
  'gold_answer': 'Provide:\n- Connection to ground\n- Main fuse\n- Residual current device\n- A lockable switch in the OFF position\nA main switch shall also be installed to power off all equipment in the robot application as an easy means for lockout.'},
 {'machine': 'UR5e Cobot',
  'question': 'What should you do before turning on the robot arm after connecting the Robot Cable to the Control Box?',
  'gold_answer': 'Plug and lock the cable from the robot into the connector at the bottom of the Control Box, then twist the connector twice to ensure it is properly locked. Do not disconnect the Robot Cable when the robot arm is turned on, and do not extend or modify the original Robot Cable.'},
 {'machine': 'UR5e Cobot',
  'question': 'Why is it important to set the robot arm mounting correctly in the installation settings?',
  'gold_answer':

## Generate — Bridgeport Mill (target: 50 pairs)

10 batches × 5 pairs per batch = 50 target.

In [13]:
mill_results = []
for i in range(10):
    batch = generate_synthetic_qa(
        client,
        document_chunks=mill_chunks,
        golden_df=mill_golden,
        machine_name="Bridgeport Mill",
    )
    mill_results.extend(batch)
    print(f"  Batch {i+1:02d}: +{len(batch)} pairs  (running total: {len(mill_results)})")

print(f"\nBridgeport Mill total: {len(mill_results)} Q&A pairs generated")

  Batch 01: +5 pairs  (running total: 5)
  Batch 02: +5 pairs  (running total: 10)
  Batch 03: +5 pairs  (running total: 15)
  Batch 04: +5 pairs  (running total: 20)
  Batch 05: +5 pairs  (running total: 25)
  Batch 06: +5 pairs  (running total: 30)
  Batch 07: +5 pairs  (running total: 35)
  Batch 08: +5 pairs  (running total: 40)
  Batch 09: +5 pairs  (running total: 45)
  Batch 10: +5 pairs  (running total: 50)

Bridgeport Mill total: 50 Q&A pairs generated


## Combine and Save

In [14]:
all_results = ur5e_results + mill_results
df_out = pd.DataFrame(all_results, columns=["machine", "question", "gold_answer"])

OUT_CSV = Path("../data/QA/synthetic_qa_cobot_mill.csv")
df_out.to_csv(OUT_CSV, index=False)

print(f"Saved {len(df_out)} Q&A pairs to {OUT_CSV}")
print()
print(df_out["machine"].value_counts().to_string())

Saved 100 Q&A pairs to ../data/QA/synthetic_qa_cobot_mill.csv

machine
UR5e Cobot         50
Bridgeport Mill    50


## Preview

In [15]:
display(df_out.head(10))

,machine,question,gold_answer
0,UR5e Cobot,What must be provided as part of the electrica...,Provide:\n- Connection to ground\n- Main fuse\...
1,UR5e Cobot,What should you do before turning on the robot...,Plug and lock the cable from the robot into th...
2,UR5e Cobot,Why is it important to set the robot arm mount...,Specifying the mounting makes the Robot arm ap...
3,UR5e Cobot,What actions can PolyScope take if the robot l...,"The selectable actions are None, Pause, or Sto..."
4,UR5e Cobot,"What are the input voltage range, input freque...",Input voltage: 90–264 VAC. Input frequency: 47...
5,UR5e Cobot,How do you configure the robot to operate with...,"In Header, tap Installation, then Safety > Har..."
6,UR5e Cobot,What safety precaution is required when the Te...,The Emergency Stop button on the Teach Pendant...
7,UR5e Cobot,"When does the Safety Checksum change, and how ...",The Safety Checksum changes when Safety Functi...
8,UR5e Cobot,"What does the MODBUS Sequential mode do, and w...",Sequential mode forces the MODBUS client to wa...
9,UR5e Cobot,What is the warning for tool analog inputs in ...,Analog Inputs are not protected against over v...
